# Validating CoLoRe simulation for CMB lensing auto

- The first and second CoLoRe simulation boxes were generated by Laura Casas, Dani L. Jarque run the rest from then on
- The notebook just uses the CMB lensing convergence $\kappa$ values of the boxes and validates it through the auto-spectrum.
    - The kappa values (`out_kappa_z000.fits`) can be found in `./data/x_test_box/cmb_lensing/`, along with $C_{\ell}^{\kappa \kappa}$, the corresponding errorbars, and a logarithmic rebinning run with `./scripts/generate_rebinned_cls.py`.
- The notebook also compares two CoLoRe boxes.
- The notebook also compares different theory lines (linear/non-linear vs Limber/non-Limber)
- The notebook also fits for one parameter in the modelling accounting for CoLoRe finite cell size.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import camb

from scipy.stats import chi2
from scipy.optimize import minimize_scalar
from scipy.integrate import simpson
from scipy.constants import c

import sys
sys.path.append('../')

from utils.binning import hybrid_bin_cls
from utils.colore_cls import compute_cls_kk
import json

# Theory CMB lensing auto-spectrum $C_{\ell}^{\kappa \kappa}$

## setting up CAMB

In [ ]:
def load_cosmology(filename):
    with open(filename) as f:
        return json.load(f)

In [ ]:
def get_camb_results(
    cosmo_dict,
    z_pk,
    kmax=100.0,
    lmax=2500,
    nonlinear=False,
):
    """
    Read a cosmological parameter dictionary, extract cosmological parameters,
    and build CAMB results instance
    """

    # Convert to CAMB inputs

    H0 = 100.0 * cosmo_dict["h"]
    ombh2 = cosmo_dict["omega_B"] * cosmo_dict["h"]**2
    omch2 = (cosmo_dict["omega_M"] - cosmo_dict["omega_B"]) * cosmo_dict["h"]**2

    # Initial CAMB params

    pars = camb.CAMBparams()
    pars.set_cosmology(
        H0=H0,
        ombh2=ombh2,
        omch2=omch2,
    )
    pars.set_dark_energy(w=cosmo_dict["w"])
    pars.InitPower.set_params(
        ns=cosmo_dict["ns"],
        As=cosmo_dict["As"]
    )

    # Matter power settings

    pars.set_matter_power(
        redshifts=z_pk,
        kmax=kmax
    )

    # note that to get CAMB's internal calculation accurate at the 1% level at L~2000,
    # need lens_potential_accuracy=2. Increase to 4 for accurate match to the Limber calculation here
    if nonlinear:
        pars.NonLinear = camb.model.NonLinear_pk
        pars.set_for_lmax(lmax=lmax, lens_potential_accuracy=4, nonlinear=True)
    else:
        pars.set_for_lmax(lmax=lmax, lens_potential_accuracy=4, nonlinear=False)

    # Final results
    results = camb.get_results(pars)

    return results

## manual Limber integral

In [ ]:
def W_kappa(z, chi, chi_source, cosmo_params):
    '''
    CMB lensing convergence kernel
    '''
    
    # scale factor at redshift of lensing structure
    a = 1.0 / (1.0 + z)

    # cosmological parameters
    H0, om = cosmo_params
    c_kms = c * 1E-3
    
    # prefactor of the kernel
    W_prefactor = (1.5) * (H0/c_kms)**2 * om * (chi_source - chi)/chi_source
    W_geom = chi / a
    
    return W_prefactor * W_geom

In [ ]:
def limber(
        z, 
        ell, 
        kmax,
        W1,
        chi,
        Hz,
        pk_interp, 
        kp, # default fitted smoothing scale in (1/Mpc)
):
    '''
    Limber integrand for CMB lensing auto-spectrum.
    '''

    # wavenumber of Limber approximation
    k = (ell + 0.5) / chi

    # jacobian dchi/dz of the transformation integrating over z instead of chi
    c_kms = c * 1e-3

    # avoid extrapolation of CAMB kmax and small scales
    mask = (k > 1e-4) & (k < kmax)
    out = np.zeros_like(k)

    # compute the Limber integrand only where valid
    if np.any(mask):

        # matter power spectrum
        pk = np.zeros_like(k)
        pk[mask] = pk_interp.P(
            z[mask],
            k[mask],
            grid=False
        )
        # smoothing
        smooth = np.exp(-(k / kp)**2)

        out[mask] = (
            (c_kms / Hz[mask])
            * ((W1[mask] * W1[mask]) / chi[mask]**2)
            * pk[mask]
            * smooth[mask]
        )

    return out

In [ ]:
def get_limber_cls(
    params_path, 
    z_pk,
    kmax=100, 
    lmin=2,
    lmax=2500,
    kp=None, # default fitted smoothing scale in (1/Mpc)
):
    '''
    Compute the angular kk power spectra using the Limber approximation.
    '''
    
    # get cosmological parameters
    cosmo_dict = load_cosmology(params_path)
    z_max, z_out = cosmo_dict["z_max"], cosmo_dict["z_out"]

    # get linear and non-linear CAMB results
    results_lin = get_camb_results(
        cosmo_dict=cosmo_dict,
        z_pk=z_pk,
        kmax=kmax,
        lmax=lmax,
        nonlinear=False)
    results_nlin = get_camb_results(
        cosmo_dict=cosmo_dict,
        z_pk=z_pk,
        kmax=kmax,
        lmax=lmax,
        nonlinear=True)
    
    # get the matter power spectrum interpolators for linear and non-linear cases
    pk_interp_lin = results_lin.get_matter_power_interpolator(nonlinear=False, 
                                                      hubble_units=False, 
                                                      k_hunit=False,  
                                                      var1='delta_tot', 
                                                      var2='delta_tot')
    pk_interp_nlin = results_nlin.get_matter_power_interpolator(nonlinear=True, 
                                                         hubble_units=False, 
                                                         k_hunit=False,  
                                                         var1='delta_tot', 
                                                         var2='delta_tot')
    
    # comoving distance of the last scattering surface 
    z_star = results_lin.get_derived_params()['zstar']  # z_star doesn't match z_out until third box
    chi_star = results_lin.comoving_radial_distance(z_star)
    chi_max = results_lin.comoving_radial_distance(z_max)

    # cosmological parameters needed by kernels
    H0 = cosmo_dict["h"] * 100
    om = cosmo_dict["omega_M"]
    cosmo_params = (H0, om)

    
    #store the cls from linear and nonlinear limber approximation integration
    # redshift integration up to zmax
    ells = np.arange(lmin, lmax + 1)
    cls_lin_zs_zs = np.zeros_like(ells, dtype=np.float64)
    cls_nlin_zs_zs = np.zeros_like(ells, dtype=np.float64)
    cls_lin_zmax_zs = np.zeros_like(ells, dtype=np.float64)
    cls_nlin_zmax_zs = np.zeros_like(ells, dtype=np.float64)
    #cls_lin_zmax_zmax = np.zeros_like(ells, dtype=np.float64)
    #cls_nlin_zmax_zmax = np.zeros_like(ells, dtype=np.float64)

    # precompute the z, chi(z) and H(z) grids for Limber integrand
    z_grid_zs = np.linspace(1e-4, z_star, 10000)  # more points to sample from 1e-4 to zstar
    chi_grid_zs = results_lin.comoving_radial_distance(z_grid_zs)
    Hz_grid_zs = results_lin.hubble_parameter(z_grid_zs)

    z_grid_zmax = np.linspace(1e-4, z_max, 1000)
    chi_grid_zmax = results_lin.comoving_radial_distance(z_grid_zmax)
    Hz_grid_zmax = results_lin.hubble_parameter(z_grid_zmax)

    # precompute kernels on the z grid
    W_grid_zs_zs = W_kappa(z_grid_zs, chi_grid_zs, chi_star, cosmo_params)
    W_grid_zmax_zs = W_kappa(z_grid_zmax, chi_grid_zmax, chi_star, cosmo_params)
    #W_grid_zmax_zmax = W_kappa(z_grid_zmax, chi_grid_zmax, chi_max, cosmo_params)

    # integrate Limber approximation
    for i, ell in enumerate(ells):

        integrand_lin_zs_zs = limber(
            z_grid_zs, ell, kmax, W_grid_zs_zs, chi_grid_zs, Hz_grid_zs, pk_interp_lin, kp
        )
        integrand_nlin_zs_zs = limber(
            z_grid_zs, ell, kmax, W_grid_zs_zs, chi_grid_zs, Hz_grid_zs, pk_interp_nlin, kp
        )
        integrand_lin_zmax_zs = limber(
            z_grid_zmax, ell, kmax, W_grid_zmax_zs, chi_grid_zmax, Hz_grid_zmax, pk_interp_lin, kp
        )
        integrand_nlin_zmax_zs = limber(
            z_grid_zmax, ell, kmax, W_grid_zmax_zs, chi_grid_zmax, Hz_grid_zmax, pk_interp_nlin, kp
        )
        cls_lin_zs_zs[i] = simpson(integrand_lin_zs_zs, x=z_grid_zs)
        cls_nlin_zs_zs[i] = simpson(integrand_nlin_zs_zs, x=z_grid_zs)
        cls_lin_zmax_zs[i] = simpson(integrand_lin_zmax_zs, x=z_grid_zmax)
        cls_nlin_zmax_zs[i] = simpson(integrand_nlin_zmax_zs, x=z_grid_zmax)

    return {
        'ells': ells,
        'results_lin': results_lin,
        'results_nlin': results_nlin,
        'cls_kk_lin_zmax_zs': cls_lin_zmax_zs,
        'cls_kk_nlin_zmax_zs': cls_nlin_zmax_zs,
        'cls_kk_lin_zs_zs': cls_lin_zs_zs,
        'cls_kk_nlin_zs_zs': cls_nlin_zs_zs,
        'z_max': float(z_max),
        'z_out': float(z_out),
        'z_star': z_star,
    }


In [ ]:
# set the redshift range for the boltzmann solver
zmin, zmax = 0, 1100
nz = 200
z_pk = zmin + np.exp(np.log(zmax - zmin + 1)*np.linspace(0, 1, nz)) - 1
z_pk = z_pk[::-1] # reverse to have higher redshift first for better interpolation at high z
lmin, lmax = 2, 2500

In [ ]:
# get the limber cls for the second and third boxes
best_kp = 0.312 # best fit smoothing scale in (1/Mpc) from fitting to the first box
box2 = get_limber_cls('../data/second_test_box/cosmo_params.json', z_pk=z_pk, kmax=100, lmin=2, lmax=2500, kp=best_kp)
box3 = get_limber_cls('../data/third_test_box/cosmo_params.json', z_pk=z_pk, kmax=100, lmin=2, lmax=2500, kp=best_kp)
box4 = get_limber_cls('../data/fourth_test_box/cosmo_params.json', z_pk=z_pk, kmax=100, lmin=2, lmax=2500, kp=best_kp)

# extract the relevant results for the second and third boxes
keys = ['ells', 'results_lin', 'results_nlin', 'cls_kk_lin_zs_zs', 'cls_kk_nlin_zs_zs', 'cls_kk_lin_zmax_zs', 'cls_kk_nlin_zmax_zs', 'z_out', 'z_star']

ells, results_lin_box2, results_nlin_box2, cl_kk_lin_box2_zs_zs, cl_kk_nlin_box2_zs_zs, cl_kk_lin_box2_zmax_zs, cl_kk_nlin_box2_zmax_zs, z_out_box2, z_star_box2 = [box2[k] for k in keys]
ells, results_lin_box3, results_nlin_box3, cl_kk_lin_box3_zs_zs, cl_kk_nlin_box3_zs_zs, cl_kk_lin_box3_zmax_zs, cl_kk_nlin_box3_zmax_zs, z_out_box3, z_star_box3 = [box3[k] for k in keys]
ells, results_lin_box4, results_nlin_box4, cl_kk_lin_box4_zs_zs, cl_kk_nlin_box4_zs_zs, cl_kk_lin_box4_zmax_zs, cl_kk_nlin_box4_zmax_zs, z_out_box4, z_star_box4 = [box4[k] for k in keys]
z_max = box2['z_max']

## full sky (non-Limber) integral

In [ ]:
def get_camb_cls(results, lmin, lmax):
    '''
    Get the CAMB kappa power spectrum Cl^kappakappa from the CAMB results object
    '''
    camb_cls = results.get_lens_potential_cls(lmax=lmax, raw_cl=True)
    camb_cls = camb_cls[lmin:, 0]
    ells = np.arange(lmin, lmax+1)
    # Convert from C_ell^phiphi to C_ell^kappakappa
    camb_cls = (ells * (ells + 1) / 2 )**2 * camb_cls  

    return camb_cls

In [ ]:
# get the CAMB cls for the corresponding box
cl_kk_camb_lin_box3 = get_camb_cls(results_lin_box3, lmin, lmax)
cl_kk_camb_nlin_box3 = get_camb_cls(results_nlin_box3, lmin, lmax) 

# Simulation CMB lensing auto-spectrum $C_{\ell}^{\kappa \kappa}$

In [ ]:
def get_colore_cls(box_filename):
    '''
    Load the Colore kappa power spectrum from a text file
    '''

    # load the data from the .npz file
    data = np.load(box_filename)
    ells, cl, err = data['ells'], data['cls'], data['err']

    return ells, cl, err

In [ ]:
# get the colore cls
#ells_sim_log, cl_kk_sim_box1, err_kk_box1 = get_colore_cls('../data/first_test_box/cmb_lensing/cl_kk_log.npz')
ells_sim_log, cl_kk_sim_box2, err_kk_box2 = get_colore_cls('../data/second_test_box/cmb_lensing/cl_kk_log.npz')
ells_sim_log, cl_kk_sim_box3, err_kk_box3 = get_colore_cls('../data/third_test_box/cmb_lensing/cl_kk_log.npz')
ells_sim_log, cl_kk_sim_box4, err_kk_box4 = get_colore_cls('../data/third_test_box/cmb_lensing/cl_kk_log.npz')
nbins = len(ells_sim_log)

# filter out NaN values which may arise from empty bins in the rebinning process
mask = np.isfinite(ells_sim_log)
ells_sim_log = ells_sim_log[mask]
#cl_kk_sim_box1, err_kk_box1 = [array[mask] for array in [cl_kk_sim_box1, err_kk_box1]]
#cl_kk_sim_box2, err_kk_box2 = [array[mask] for array in [cl_kk_sim_box2, err_kk_box2]]
cl_kk_sim_box3, err_kk_box3 = [array[mask] for array in [cl_kk_sim_box3, err_kk_box3]]
cl_kk_sim_box4, err_kk_box4 = [array[mask] for array in [cl_kk_sim_box4, err_kk_box4]]

# determine the lmin and lmax from the Colore data to ensure consistent binning for residuals
lmin_colore = lmin
lmax_colore = np.load(f'../data/third_test_box/cmb_lensing/cl_kk.npz')['ells'].max()

# Plots

## latest CoLoRe box analysis and $\chi^2$ test

### functions

In [ ]:
def plot_cls_comparison_noise(ells_sim, cl_kk_sim, err_kk_sim, 
                              ells, cl_kk_lin, cl_kk_lin_rebin, z_out, z_star, suptitle, box_label,
                              ells_sim_raw=None, cl_kk_sim_raw=None, err_kk_sim_raw=None, box_label_raw=None):

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 5), height_ratios=[2.5, 1], dpi=120, sharex=True, gridspec_kw={'hspace':0.05})
    
    for ax in (ax1, ax2):
        ax.tick_params(
            which='both',
            direction='in',
            top=True,
            right=True
        )

    # compute the reduced chi-squared between the rebinned linear limber and the rebinned CoLoRe cls
    chi2 = np.sum((cl_kk_sim - cl_kk_lin_rebin)**2 / err_kk_sim**2)
    chi2_nu = chi2 / (len(ells_sim))

    # plot the rebinned CoLoRe Cls with error bars
    ax1.errorbar(ells_sim, cl_kk_sim, yerr=err_kk_sim, fmt='k.', ms=5.0,
                 elinewidth=0.8, capsize=2.0, ecolor='k',
                 label=rf'CoLoRe_{box_label}')
    ax1.scatter(0, 0 , s=0, label=rf'$\chi^2_\nu = {chi2_nu:.2f} \ \ (k_p= {best_kp:.3f})$')

    if cl_kk_sim_raw is not None:
        # plot the rebinned CoLoRe Cls with error bars
        ax1.errorbar(ells_sim_raw, cl_kk_sim_raw, yerr=err_kk_sim_raw, fmt='C1.', ms=5.0,
                 elinewidth=0.8, capsize=2.0, ecolor='C1',
                label=rf'CoLoRe_{box_label_raw}')
        # plot the relative error between the rebinned linear limber and the rebinned CoLoRe cls
        chi2_bin_raw = (cl_kk_sim_raw - cl_kk_lin_rebin) / err_kk_sim_raw
        ax2.errorbar(ells_sim, chi2_bin_raw, fmt='C1.', ms=5.0, mec='C1')

    # manual limber integration full redshift
    ax1.plot(ells, cl_kk_lin, c='C0', ls='-', label=rf'Limber_linear ($z_{{\rm cut}} = {z_max}, \ z_s={int(z_star)}$)')

    # shade most sensitive region to the analysis
    lmin_fit, lmax_fit = 20, 1500
    ax1.axvspan(lmin_fit, lmax_fit, color='gray', alpha=0.1, label='Fitting region')

    ax1.set_xscale('log')
    ax1.set_yscale('log')
    ax1.set_xlim(1.5, lmax)
    ax1.set_ylabel(r'$C_{L}^{\kappa \kappa}$')
    ax1.legend()
    ax1.grid(True, which='major', alpha=0.25)

    # plot the relative error between the rebinned linear limber and the rebinned CoLoRe cls
    chi2_bin = (cl_kk_sim - cl_kk_lin_rebin) / err_kk_sim

    # plot the relative error for low multipoles in the main panel
    ax2.errorbar(ells_sim, chi2_bin, fmt='k.', ms=5.0, mec='k')
    ax2.axhspan(-1, 1, color='C0', alpha=0.12)
    ax2.axvspan(lmin_fit, lmax_fit, color='gray', alpha=0.15, label='Fitting region')
    ax2.axhline(0, color='0.35', ls='--', lw=1.0, zorder=0) 
    ax2.set_xscale('log')
    ax2.set_xlabel(r'Multipole $L$')
    ax2.set_ylabel(r'$\Delta C_L^{kk}/\sigma$')
    ax2.set_ylim(-5, 5)


    fig.suptitle(suptitle, y=0.93)

    fig.savefig(f'plots/cls_kk_{box_label}.png', bbox_inches='tight')
    fig.savefig(f'plots/cls_kk_{box_label}.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
def plot_cls_comparison(ells_sim, cl_kk_sim, err_kk_sim, ells, cl_kk_lin, cl_kk_lin_rebin, z_out, z_star, box_label, plot_title=''):

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 5), height_ratios=[2.5, 1], dpi=120, sharex=True, gridspec_kw={'hspace':0.05})

    for ax in (ax1, ax2):
        ax.tick_params(
            which='both',
            direction='in',
            top=True,
            right=True
        )

    # compute the reduced chi-squared between the rebinned linear limber and the rebinned CoLoRe cls
    chi2 = np.sum((cl_kk_sim - cl_kk_lin_rebin)**2 / err_kk_sim**2)
    chi2_nu = chi2 / (len(ells_sim))

    # plot the rebinned CoLoRe Cls with error bars
    ax1.errorbar(ells_sim, cl_kk_sim, yerr=err_kk_sim, fmt='k.', ms=5.0,
                 elinewidth=0.8, capsize=2.0, ecolor='k',
                 label=rf'CoLoRe_{box_label} $(z_{{\rm max}} = {z_max}, \ z_s={int(z_out)})$')
    ax1.scatter(0, 0 , s=0, label=rf'$\chi^2_\nu = {chi2_nu:.2f} \ (k_p= {best_kp:.3f})$')

    # manual limber integration full redshift
    ax1.plot(ells, cl_kk_lin, c='C0', ls='-', label=rf'Limber_linear ($z_{{\rm cut}} = {z_max}, \ z_s={int(z_star)}$)')

    # shade most sensitive region to the analysis
    lmin_fit, lmax_fit = 20, 1000
    ax1.axvspan(lmin_fit, lmax_fit, color='gray', alpha=0.1, label='Fitting region')

    ax1.set_xscale('log')
    ax1.set_yscale('log')
    ax1.set_xlim(1.5, 3000)
    ax1.set_ylabel(r'$C_{L}^{\kappa \kappa}$')
    ax1.legend()
    ax1.grid(True, which='major', alpha=0.25)


    # significance of the rebinned linear limber and the rebinned CoLoRe cls
    chi2_bin = (cl_kk_sim - cl_kk_lin_rebin) / err_kk_sim

    # plot the significance
    ax2.errorbar(ells_sim, chi2_bin, fmt='k.', ms=5.0, mec='k')
    ax2.axhspan(-1, 1, color='C0', alpha=0.12)
    ax2.axvspan(lmin_fit, lmax_fit, color='gray', alpha=0.15, label='Fitting region')
    ax2.axhline(0, color='0.35', ls='--', lw=1.0, zorder=0) 
    ax2.set_xscale('log')
    ax2.set_xlabel(r'Multipole $L$')
    ax2.set_ylabel(r'$\Delta C_L^{kk}/\sigma$')
    ax2.set_ylim(-5, 5)

    fig.suptitle(plot_title, y=1.001)
    fig.savefig(f'plots/cls_kk_{box_label}.png', bbox_inches='tight')
    fig.savefig(f'plots/cls_kk_{box_label}.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
def plot_cumulative_chi2(ells_colore, cl_kk_colore, err_kk, cl_kk_lin_rebin, box_label):

    # compute the cumulative chi-square and p-values for each multipole bin
    chi2_per_bin = ((cl_kk_colore - cl_kk_lin_rebin) / err_kk)**2
    cumulative_chi2 = np.nancumsum(chi2_per_bin)
    cumulative_dof = np.arange(1, len(cumulative_chi2) + 1)
    p_values = 1 - chi2.cdf(cumulative_chi2, df=cumulative_dof)

    # Plotting the cumulative chi-square
    fig, ax1 = plt.subplots(figsize=(6, 4), dpi=120)
    ax2 = ax1.twinx()  # Create a second y-axis for p-values

    # Plot your cumulative chi-square
    ax1.plot(ells_colore, cumulative_chi2, marker='.', linestyle='-', color='C0', label=rf'Cumulative $\chi^2$ ({box_label})')
    ax2.plot(ells_colore, p_values, marker='.', linestyle='-', color='C1', label=r'Cumulative $p$-value', alpha=0.3)

    # Plot the expected theoretical baseline (Expected chi^2 = dof)
    ax1.plot(ells_colore, cumulative_dof, linestyle='--', color='black', label=r'Expected $\chi^2$ (Perfect Fit)')

    # Formatting the plot
    ax1.set_xscale('log')
    ax1.set_yscale('linear') # A log scale on the y-axis is highly recommended since the max value is ~8000
    ax1.set_ylim(0, 100)
    ax1.set_xlabel('$L$')
    ax1.set_ylabel(r'Cumulative $\chi^2$')
    ax2.set_ylabel(r'Cumulative $p$-value')
    ax1.set_title(r'Cumulative $\chi^2$ vs. Multipole $L$')
    ax1.legend(loc='upper left')
    ax2.legend(loc='upper right')

    fig.savefig(f'plots/cls_kk_{box_label}_cumulative_chi2.png', bbox_inches='tight')  
    fig.savefig(f'plots/cls_kk_{box_label}_cumulative_chi2.pdf', bbox_inches='tight')  

    plt.show()

### noiseless data

In [ ]:
# rebin the CAMB linear and non-linear cls to match the number of bins in the Colore data for residuals
_, cl_kk_lin_box2_zmax_zs_rebin, _ = hybrid_bin_cls(ells, cl_kk_lin_box2_zmax_zs, nbins=nbins, ell_min=lmin_colore, ell_max=lmax_colore)
_, cl_kk_lin_box3_zmax_zs_rebin, _ = hybrid_bin_cls(ells, cl_kk_lin_box3_zmax_zs, nbins=nbins, ell_min=lmin_colore, ell_max=lmax_colore)
_, cl_kk_lin_box4_zmax_zs_rebin, _ = hybrid_bin_cls(ells, cl_kk_lin_box4_zmax_zs, nbins=nbins, ell_min=lmin_colore, ell_max=lmax_colore)

# filter out NaN values which may arise from empty bins in the rebinning process
cl_kk_lin_box2_zmax_zs_rebin = cl_kk_lin_box2_zmax_zs_rebin[mask]
cl_kk_lin_box3_zmax_zs_rebin = cl_kk_lin_box3_zmax_zs_rebin[mask]
cl_kk_lin_box4_zmax_zs_rebin = cl_kk_lin_box4_zmax_zs_rebin[mask]

In [ ]:
cl_kk_lin_box4_zmax_zs

In [ ]:
plot_cls_comparison(ells_sim_log, cl_kk_sim_box4, err_kk_box4, ells, cl_kk_lin_box4_zmax_zs, 
                    cl_kk_lin_box4_zmax_zs_rebin, z_out_box4, z_star_box4, 
                    box_label='fourth_test_box',
                    plot_title=f'CMB Lensing auto-spectrum:\n Linear Theory vs. CoLoRe box')

In [ ]:
plot_cumulative_chi2(ells_sim_log, cl_kk_sim_box3, err_kk_box3, cl_kk_lin_box3_zmax_zs_rebin, 
                     box_label='third_test_box')

### noisy data

In [ ]:
# get the noise spectrum
infile = '../data/third_test_box/cmb_lensing/nlkk_v3_1_0deproj0_SENS2_fsky0p4_it_lT30-3000_lP30-5000.dat'
outfile = '../data/third_test_box/cmb_lensing/nlkk_SO_MV.dat'
ells_nl, nl_mv = np.loadtxt(infile, usecols=(0, 7), unpack=True)

ells_nl = np.insert(ells_nl, [0, 0], [0, 1])
nl_mv = np.insert(nl_mv, [0, 0], [0.0, 0.0])

np.savetxt(
    outfile,
    np.column_stack([ells_nl, nl_mv]),
    fmt='%.6e',
    header='ells, N_lensing_MV (all)'
)

In [ ]:
# get the noisy colore cls
kappa_file = '../data/third_test_box/cmb_lensing/out_kappa_z000.fits'
ells_sim, cl_kk_sim_box3_denoise, err_kk_box3_denoise, cl_kk_sim_box3_raw = compute_cls_kk(kappa_file, outfile, return_raw_cls=True)
ells_sim_log, cl_kk_sim_box3_denoise, err_kk_box3_denoise = hybrid_bin_cls(ells_sim, cl_kk_sim_box3_denoise, err=err_kk_box3_denoise, nbins=nbins, ell_min=lmin_colore, ell_max=lmax_colore)
ells_sim_log, cl_kk_sim_box3_raw, _ = hybrid_bin_cls(ells_sim, cl_kk_sim_box3_raw, nbins=nbins, ell_min=lmin_colore, ell_max=lmax_colore)

# filter out NaN values which may arise from empty bins in the rebinning process
mask = np.isfinite(ells_sim_log)
ells_sim_log = ells_sim_log[mask]
cl_kk_box3_noise, err_kk_box3_noise = [array[mask] for array in [cl_kk_sim_box3_denoise, err_kk_box3_denoise]]

In [ ]:
plot_cls_comparison_noise(ells_sim_log, cl_kk_sim_box3_denoise, err_kk_box3_noise, ells, cl_kk_lin_box3_zmax_zs, 
                    cl_kk_lin_box3_zmax_zs_rebin, z_out_box3, z_star_box3, 
                    suptitle='noisy CMB lensing auto',
                    box_label='third_box_denoise',
                    ells_sim_raw=ells_sim_log, cl_kk_sim_raw=cl_kk_sim_box3_raw, err_kk_sim_raw=err_kk_box3_denoise, box_label_raw='third_box_raw')  

In [ ]:
plot_cls_comparison_noise(ells_sim_log, cl_kk_sim_box3, err_kk_box3, ells, cl_kk_lin_box3_zmax_zs, 
                    cl_kk_lin_box3_zmax_zs_rebin, z_out_box3, z_star_box3, 
                    suptitle='noiseless CMB lensing auto',
                    box_label='third_box_noiseless')  

## comparing CoLoRe boxes

In [ ]:
def plot_compare_boxes(ells_colore, cl_kk_colore_box1, cl_kk_colore_box2, err_kk_1, err_kk_2, 
                       ells, cl_kk_lin1, cl_kk_lin2, cl_kk_lin1_rebin, cl_kk_lin2_rebin,
                       z_max, z_out_box1, z_out_box2, z_star_box1, z_star_box2, box_label1, box_label2):

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 5), height_ratios=[2.5, 1], dpi=120, sharex=True, gridspec_kw={'hspace':0.05})

    # plot the rebinned CoLoRe Cls with error bars
    ax1.errorbar(ells_colore, cl_kk_colore_box1, yerr=err_kk_1, elinewidth=0.7, capsize=1.0, fmt='C1.', markersize=5.0, label=rf'CoLoRe_{box_label1} ($z_{{\rm max}} = {z_max}, \ z_s={int(z_out_box1)}$)')
    ax1.errorbar(ells_colore, cl_kk_colore_box2, yerr=err_kk_2, elinewidth=0.7, capsize=1.0, fmt='C0.', markersize=5.0, label=rf'CoLoRe_{box_label2} ($z_{{\rm max}} = {z_max}, \ z_s={int(z_out_box2)}$)')

    # manual limber integration full redshift
    ax1.plot(ells, cl_kk_lin1, c='C1', ls='-', label=rf'Limber_linear ($z_{{\rm cut}} = {z_max}, \ z_s={int(z_star_box1)}$)')
    ax1.plot(ells, cl_kk_lin2, c='C0', ls='-', label=rf'Limber_linear ($z_{{\rm cut}} = {z_max}, \ z_s={int(z_star_box2)}$)')

    ax1.set_xscale('log')
    ax1.set_yscale('log')
    ax1.set_xlim(1.5, lmax)
    ax1.set_ylabel(r'$C_{L}^{\kappa \kappa}$')
    ax1.legend()
    ax1.grid(True, which='major', alpha=0.25)

    # plot the relative error between the rebinned linear limber and the rebinned CoLoRe cls
    ax2.errorbar(ells_colore, (cl_kk_colore_box1-cl_kk_lin1_rebin)/cl_kk_lin1_rebin, 
                yerr=err_kk_1/cl_kk_lin1_rebin, capsize=1.5, elinewidth=0.7, fmt='C0.-', markersize=5.0)
    ax2.errorbar(ells_colore, (cl_kk_colore_box2-cl_kk_lin2_rebin)/cl_kk_lin2_rebin, 
                yerr=err_kk_2/cl_kk_lin2_rebin, capsize=1.5, elinewidth=0.7, fmt='C1.-', markersize=5.0)
    ax2.axhline(0, c='k', ls='--', lw=0.7)
    ax2.set_xscale('log')
    ax2.set_xlabel(r'$L$')
    ax2.set_ylabel('relative error')
    ax2.set_ylim(-1.0, 1.0)

    fig.suptitle('CMB lensing autospectrum: CoLoRe boxes comparison', y=0.93)

    fig.savefig(f'plots/cls_kk_boxes_comparison.png', bbox_inches='tight')
    fig.savefig(f'plots/cls_kk_boxes_comparison.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
plot_compare_boxes(
    ells_sim_log, 
    cl_kk_sim_box2, cl_kk_sim_box3, 
    err_kk_box2, err_kk_box3, 
    ells,
    cl_kk_lin_box2_zmax_zs, cl_kk_lin_box3_zmax_zs, 
    cl_kk_lin_box2_zmax_zs_rebin, cl_kk_lin_box3_zmax_zs_rebin,
    z_max, z_out_box2, z_out_box3, z_star_box2, z_star_box3, 
    box_label1='second_box', 
    box_label2='third_box')

## comparing theory

In [ ]:
def plot_theory_cls(ells, cl_kk_lin_zs_zs, cl_kk_nlin_zs_zs, cl_kk_camb_lin, cl_kk_camb_nlin, z_star):
    '''
    Plot the theoretical CMB lensing auto-spectrum from manual Limber integration and CAMB fullsky integration 
    for both linear and non-linear matter power spectra
    '''

    plt.figure(figsize=(6, 4), dpi=120)

    # manual limber integration full redshift
    plt.loglog(ells, cl_kk_lin_zs_zs, c='C0', ls='-', label=rf'Limber_lin ($z_{{\rm cut}} = z_s={int(z_star)}$)')
    plt.loglog(ells, cl_kk_nlin_zs_zs, c='C0', ls='--', label=rf'Limber_nonlin ($z_{{\rm cut}} = z_s={int(z_star)}$)')

    # full sky integration full redshift
    plt.loglog(ells, cl_kk_camb_lin, c='C1', ls='-', label=rf'nonLimber_lin ($z_{{\rm cut}} = z_s={int(z_star)}$)')
    plt.loglog(ells, cl_kk_camb_nlin, c='C1', ls='--', label=rf'nonLimber_nonlin ($z_{{\rm cut}} = z_s={int(z_star)}$)')

    plt.grid(which='major', alpha=0.25)
    plt.xlim(1.5, lmax)
    plt.ylim(8e-11, 4e-7)
    plt.xlabel(r'$L$')
    plt.ylabel(r'$C_{L}^{\kappa \kappa}$')

    plt.title('CMB lensing autospectrum: \n limber/non-Limber, linear/non-linear', y=1.05)
    plt.legend()

    plt.savefig(f'plots/cls_kk_theory_comparison.png', bbox_inches='tight')
    plt.savefig(f'plots/cls_kk_theory_comparison.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
plot_theory_cls(ells, cl_kk_lin_box3_zs_zs, cl_kk_nlin_box3_zs_zs, cl_kk_camb_lin_box3, cl_kk_camb_nlin_box3, z_star_box3)

In [ ]:
np.savez("cl_kk_lin_limber_zcut", ell=ells, cls=cl_kk_lin_box3_zmax_zs)
np.savez("cl_kk_lin_nonlimber", ell=ells, cls=cl_kk_camb_lin_box3)

# Best-fit results

## fuctions

In [ ]:
def get_limber_cls_fit(
    params_path, 
    z_pk,
    kmax=100, 
    lmin=2,
    lmax=2500,
    kp=None, # default fitted smoothing scale in (1/Mpc)
):
    '''
    Compute the angular kk power spectra for the fitting using the Limber approximation.d
    '''
    
    # get cosmological parameters
    cosmo_dict = load_cosmology(params_path)
    z_max, z_out = cosmo_dict["z_max"], cosmo_dict["z_out"]

    # get linear and non-linear CAMB results
    results_lin = get_camb_results(
        cosmo_dict=cosmo_dict,
        z_pk=z_pk,
        kmax=kmax,
        lmax=lmax,
        nonlinear=False)
    
    # get the matter power spectrum interpolators for linear and non-linear cases
    pk_interp_lin = results_lin.get_matter_power_interpolator(nonlinear=False, 
                                                      hubble_units=False, 
                                                      k_hunit=False,  
                                                      var1='delta_tot', 
                                                      var2='delta_tot')
    
    # comoving distance of the last scattering surface 
    z_star = results_lin.get_derived_params()['zstar']  # z_star doesn't match z_out until third box
    chi_star = results_lin.comoving_radial_distance(z_star)

    # cosmological parameters needed by kernels
    H0 = cosmo_dict["h"] * 100
    om = cosmo_dict["omega_M"]
    cosmo_params = (H0, om)

    #store the cls from linear and nonlinear limber approximation integration
    # redshift integration up to zmax
    ells = np.arange(lmin, lmax + 1)
    cls_lin_zmax_zs = np.zeros_like(ells, dtype=np.float64)

    # precompute the z, chi(z) and H(z) grids for Limber integrand
    z_grid_zmax = np.linspace(1e-4, z_max, 1000)
    chi_grid_zmax = results_lin.comoving_radial_distance(z_grid_zmax)
    Hz_grid_zmax = results_lin.hubble_parameter(z_grid_zmax)

    # precompute kernels on the z grid
    W_grid_zmax_zs = W_kappa(z_grid_zmax, chi_grid_zmax, chi_star, cosmo_params)

    # integrate Limber approximation
    for i, ell in enumerate(ells):

        integrand_lin_zmax_zs = limber(
            z_grid_zmax, ell, kmax, W_grid_zmax_zs, chi_grid_zmax, Hz_grid_zmax, pk_interp_lin, kp
        )

        cls_lin_zmax_zs[i] = simpson(integrand_lin_zmax_zs, x=z_grid_zmax)

    return {
    'ells': ells,
    'cls_kk_lin_zmax_zs': cls_lin_zmax_zs,
    }

In [ ]:
def get_fitting_cls(
    kp,
    params_path,
    z_pk,
    lmin_fit,
    lmax_fit,
    nbins_fit,
    kmax=100.0
):
    '''
    Compute rebinned theory Cls for fitting.
    '''

    # compute theory
    results = get_limber_cls_fit(
        params_path=params_path,
        z_pk=z_pk,
        kmax=kmax,
        lmin=lmin_fit,
        lmax=lmax_fit,
        kp=kp
    )

    # rebin theory
    _, cls_theory, _ = hybrid_bin_cls(
        results["ells"],
        results["cls_kk_lin_zmax_zs"],
        nbins=nbins_fit,
        ell_min=lmin_fit,
        ell_max=lmax_fit,
    )

    return cls_theory

In [ ]:
def get_chi2(
        kp, 
        params_path, 
        z_pk, 
        lmin_fit, 
        lmax_fit,
        cls_data,
        err_data
):
    '''
    Reduced chi-squared for kk fitting.
    '''

    # compute and rebin theory
    cls_theory = get_fitting_cls(
        kp=kp,
        params_path=params_path,
        z_pk=z_pk,
        lmin_fit=lmin_fit,
        lmax_fit=lmax_fit,
        nbins_fit=len(cls_data),
    )

    # remove invalid values
    mask = (
        np.isfinite(cls_data)
        & np.isfinite(cls_theory)
        & np.isfinite(err_data)
        & (err_data > 0)
    )

    cls_data = cls_data[mask]
    cls_theory = cls_theory[mask]
    err_data = err_data[mask]

    # get chi2 statistic
    chi2 = np.sum(
        ((cls_data - cls_theory) / err_data)**2
    )

    return chi2 / (len(cls_data) - 1)  # fitting 1 parameter

## minimizer

In [ ]:
# set the fitting range for the chi2 minimization
lmin_fit = 2
lmax_fit = 1000
nbins_fit = 30

# paths to the cosmological parameters and the CoLoRe Cls for the third box
params_path = '../data/third_test_box/cosmo_params.json'
box_path = '../data/third_test_box/cmb_lensing/cl_kk.npz'

In [ ]:
# load and rebin data
ells_data, cls_data, err_data = get_colore_cls(box_path)
ells_data, cls_data, err_data = hybrid_bin_cls(
        ells_data,
        cls_data,
        err_data,
        nbins=nbins_fit,
        ell_min=lmin_fit,
        ell_max=lmax_fit,
    )

In [ ]:
# perform the chi2 minimization to fit the smoothing scale for the kk spectrum
result = minimize_scalar(
    get_chi2, 
    bounds=(0.1, 1.0), 
    method='bounded', 
    args=(
        params_path, 
        z_pk, 
        lmin_fit, 
        lmax_fit,
        cls_data,
        err_data
    )
)

best_kp = result.x
chi2_best = result.fun

In [ ]:
print(f"Best-fit k_p: {best_kp:.3f} 1/Mpc")
print(f"Reduced chi-squared at best-fit k_p: {chi2_best:.3f}")

In [ ]:
def plot_chi2(kp_array, chi_array):
    '''
    Plot the chi-squared statistic as a function of k_p for the fit of the theoretical C_l^kk to the Colore data
    '''
    
    plt.figure(figsize=(6, 4), dpi=120)

    plt.plot(kp_array, chi_array, marker='.')
    plt.scatter(best_kp, chi2_best, s=50, c='red', label=rf'$\chi^2_\nu = {chi2_best:.2f}$')

    plt.axvline(
        best_kp,
        color='red',
        ls='--',
        alpha=0.7
    )

    # annotation
    plt.annotate(
        rf'$k_p = {best_kp:.3f}$',
        xy=(best_kp, chi2_best),
        xytext=(10, 25),
        textcoords='offset points',
        color='red'
    )

    plt.xlabel(r'$k_p$ [1/Mpc]')
    plt.ylabel(r'$\chi^2_\nu$')
    plt.title(r'CMB Lensing Auto-Spectrum $\chi^2$-Fit of $k_p$')
    plt.savefig('./plots/kp_fit_chi2_kk.png', dpi=150, bbox_inches='tight')
    plt.savefig('./plots/kp_fit_chi2_kk.pdf', dpi=150, bbox_inches='tight')
    plt.legend()
    plt.show()

In [ ]:
kp_array = np.linspace(0.1, 0.5, 20)
chi2_array = [get_chi2(kp, params_path, z_pk, lmin_fit, lmax_fit, cls_data, err_data) for kp in kp_array]

In [ ]:
plot_chi2(kp_array, chi2_array)